In [ ]:
"""
  Content Filter: Abuse + Frustration Detection
  - Removes abusive queries (harassment, hate, threats)
  - Removes user frustrations (complaints, venting, negativity)
  - Keeps only clean queries suitable for autocomplete training
  """

  import torch
  import torch.nn.functional as F
  from transformers import AutoTokenizer, AutoModelForCausalLM
  import pandas as pd
  import numpy as np
  from typing import List, Dict
  from tqdm import tqdm
  from dataclasses import dataclass
  from enum import Enum
  import json
  import re
  import math
  import time
  import warnings
  warnings.filterwarnings("ignore")


  # =============================================================================
  # 1. CLASSIFICATION LABELS
  # =============================================================================

  class QueryLabel(Enum):
      CLEAN = "clean"                    # Good for autocomplete ✅
      FRUSTRATION = "frustration"        # User venting, complaints ❌
      MILD_INAPPROPRIATE = "mild"        # Borderline content ❌
      ABUSIVE = "abusive"                # Harassment, insults ❌
      SEVERE = "severe"                  # Hate speech, threats ❌


  @dataclass
  class ContentAnalysis:
      query: str
      label: str
      category: str
      confidence: float
      keep: bool              # True = good for training
      explanation: str

      def to_dict(self) -> Dict:
          return {
              "query": self.query,
              "label": self.label,
              "category": self.category,
              "confidence": self.confidence,
              "keep": self.keep,
              "explanation": self.explanation
          }


  # =============================================================================
  # 2. FEW-SHOT PROMPT WITH FRUSTRATION EXAMPLES
  # =============================================================================

  CONTENT_FILTER_PROMPT = '''You are a content quality filter for autocomplete training data. 
  Classify each query into one of these categories:

  ## LABELS:
  - CLEAN: Appropriate query, good for autocomplete training
  - FRUSTRATION: User complaints, venting, negative emotions (NOT for training)
  - MILD: Borderline inappropriate, unprofessional language
  - ABUSIVE: Insults, harassment, personal attacks
  - SEVERE: Hate speech, threats, dangerous content, slurs

  ## CATEGORIES:
  - none: Clean query
  - complaint: User expressing dissatisfaction
  - venting: Emotional outburst, frustration
  - negativity: Pessimistic, unhelpful queries
  - profanity: Swear words
  - harassment: Personal attacks
  - hate_speech: Discrimination, slurs
  - threats: Violence, harm
  - dangerous: Illegal, harmful requests

  ## EXAMPLES:

  Query: "what is my account balance"
  {"label": "clean", "category": "none", "confidence": 0.99, "explanation": "Standard banking query"}

  Query: "transfer money to savings"
  {"label": "clean", "category": "none", "confidence": 0.98, "explanation": "Normal transaction request"}

  Query: "check my credit card rewards"
  {"label": "clean", "category": "none", "confidence": 0.97, "explanation": "Legitimate inquiry"}

  Query: "why is this app so slow"
  {"label": "frustration", "category": "complaint", "confidence": 0.92, "explanation": "User complaint about performance"}

  Query: "this is so frustrating"
  {"label": "frustration", "category": "venting", "confidence": 0.95, "explanation": "User expressing frustration"}

  Query: "ugh nothing works here"
  {"label": "frustration", "category": "venting", "confidence": 0.90, "explanation": "Negative venting"}

  Query: "your app sucks"
  {"label": "frustration", "category": "complaint", "confidence": 0.88, "explanation": "Negative feedback, not personal attack"}

  Query: "I hate this bank"
  {"label": "frustration", "category": "negativity", "confidence": 0.91, "explanation": "Strong negative sentiment"}

  Query: "worst service ever"
  {"label": "frustration", "category": "complaint", "confidence": 0.89, "explanation": "Complaint, not abuse"}

  Query: "why can't you people do anything right"
  {"label": "mild", "category": "complaint", "confidence": 0.85, "explanation": "Borderline - frustration with slight hostility"}

  Query: "this is so damn annoying"
  {"label": "mild", "category": "profanity", "confidence": 0.82, "explanation": "Mild profanity with frustration"}

  Query: "you people are idiots"
  {"label": "abusive", "category": "harassment", "confidence": 0.93, "explanation": "Direct insult, personal attack"}

  Query: "incompetent morons"
  {"label": "abusive", "category": "harassment", "confidence": 0.94, "explanation": "Derogatory personal attack"}

  Query: "f*** this stupid app"
  {"label": "abusive", "category": "profanity", "confidence": 0.91, "explanation": "Strong profanity"}

  Query: "I'll sue you all"
  {"label": "abusive", "category": "threats", "confidence": 0.87, "explanation": "Legal threat"}

  Query: "you [racial slur] employees"
  {"label": "severe", "category": "hate_speech", "confidence": 0.98, "explanation": "Racial slur"}

  Query: "women shouldn't handle money"
  {"label": "severe", "category": "hate_speech", "confidence": 0.96, "explanation": "Sexist discrimination"}

  Query: "I'll kill whoever did this"
  {"label": "severe", "category": "threats", "confidence": 0.97, "explanation": "Violence threat"}

  Query: "how to hack bank accounts"
  {"label": "severe", "category": "dangerous", "confidence": 0.95, "explanation": "Illegal activity request"}

  ## YOUR TASK:
  Analyze this query. Return ONLY valid JSON:
  {"label": "clean|frustration|mild|abusive|severe", "category": "category_name", "confidence": 0.0-1.0, "explanation": "brief reason"}

  Query: "{query}"
  JSON:'''


  # Shorter efficient prompt
  EFFICIENT_FILTER_PROMPT = '''Classify query for autocomplete training data.

  LABELS:
  - clean: Good for training
  - frustration: Complaints, venting, negativity (remove)
  - mild: Borderline inappropriate (remove)
  - abusive: Insults, harassment (remove)
  - severe: Hate speech, threats (remove)

  Examples:
  "check my balance" → {"label":"clean","category":"none","confidence":0.99}
  "why is this so slow" → {"label":"frustration","category":"complaint","confidence":0.92}
  "this app sucks" → {"label":"frustration","category":"complaint","confidence":0.88}
  "ugh so frustrating" → {"label":"frustration","category":"venting","confidence":0.91}
  "I hate this bank" → {"label":"frustration","category":"negativity","confidence":0.90}
  "you idiots" → {"label":"abusive","category":"harassment","confidence":0.94}
  "[slur] employees" → {"label":"severe","category":"hate_speech","confidence":0.97}

  Query: "{query}"
  JSON:'''


  # =============================================================================
  # 3. CONTENT FILTER CLASS
  # =============================================================================

  class ContentFilter:
      """
      Filters both abuse AND frustration from queries.
      Only keeps clean queries suitable for autocomplete training.
      """

      def __init__(
          self,
          model_name: str = "google/gemma-3-12b-it",
          prompt_type: str = "efficient",  # "full" or "efficient"
          batch_size: int = 8
      ):
          print(f"Loading {model_name}...")

          self.batch_size = batch_size
          self.prompt_template = EFFICIENT_FILTER_PROMPT if prompt_type == "efficient" else CONTENT_FILTER_PROMPT

          # Load tokenizer
          self.tokenizer = AutoTokenizer.from_pretrained(model_name)
          self.tokenizer.padding_side = "left"
          if self.tokenizer.pad_token is None:
              self.tokenizer.pad_token = self.tokenizer.eos_token

          # Load model
          self.model = AutoModelForCausalLM.from_pretrained(
              model_name,
              torch_dtype=torch.bfloat16,
              device_map="auto",
              attn_implementation="flash_attention_2",
              low_cpu_mem_usage=True,
          )
          self.model.eval()

          # Labels to KEEP (only clean)
          self.keep_labels = {"clean"}

          # Labels to REMOVE
          self.remove_labels = {"frustration", "mild", "abusive", "severe"}

          print(f"Model loaded. GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
          print(f"Keep labels: {self.keep_labels}")
          print(f"Remove labels: {self.remove_labels}")

      def _parse_response(self, response: str, query: str) -> ContentAnalysis:
          """Parse model response."""

          try:
              json_match = re.search(r'\{[^}]+\}', response, re.DOTALL)
              if json_match:
                  data = json.loads(json_match.group())

                  label = data.get("label", "clean").lower().strip()
                  # Normalize labels
                  if label in ["frustration", "complaint", "venting", "negativity"]:
                      label = "frustration"
                  elif label in ["mild", "borderline", "inappropriate"]:
                      label = "mild"
                  elif label in ["abusive", "harassment", "abuse"]:
                      label = "abusive"
                  elif label in ["severe", "hate", "threat", "dangerous"]:
                      label = "severe"
                  elif label not in self.keep_labels and label not in self.remove_labels:
                      label = "clean"

                  category = data.get("category", "none")
                  confidence = float(data.get("confidence", 0.8))
                  confidence = max(0.0, min(1.0, confidence))
                  explanation = str(data.get("explanation", ""))[:150]

                  keep = label in self.keep_labels

                  return ContentAnalysis(
                      query=query,
                      label=label,
                      category=category,
                      confidence=confidence,
                      keep=keep,
                      explanation=explanation
                  )

          except (json.JSONDecodeError, KeyError, TypeError, ValueError):
              pass

          # Fallback: pattern-based detection
          query_lower = query.lower()
          response_lower = response.lower()

          # Frustration patterns
          frustration_patterns = [
              r'\b(frustrat|annoying|annoyed|ugh|argh|sigh)\b',
              r'\b(hate|hating|worst|terrible|horrible|awful)\b',
              r'\b(sucks?|useless|pointless|waste)\b',
              r'\b(why (is|does|can\'t|won\'t|isn\'t))\b.*\??\s*$',
              r'\b(nothing works|doesn\'t work|not working|broken)\b',
              r'\b(so slow|too slow|taking forever)\b',
              r'\b(disappointed|disappointing|letdown)\b',
              r'\b(can\'t believe|unbelievable|ridiculous)\b',
          ]

          for pattern in frustration_patterns:
              if re.search(pattern, query_lower):
                  return ContentAnalysis(
                      query=query, label="frustration", category="detected_pattern",
                      confidence=0.75, keep=False, explanation="Frustration pattern detected"
                  )

          # Abuse patterns
          abuse_patterns = [
              r'\b(idiot|stupid|moron|dumb|incompetent)\b',
              r'\b(hate you|f+u+c+k|s+h+i+t|damn|crap)\b',
              r'\b(kill|die|death|burn|destroy)\b',
          ]

          for pattern in abuse_patterns:
              if re.search(pattern, query_lower):
                  return ContentAnalysis(
                      query=query, label="abusive", category="detected_pattern",
                      confidence=0.80, keep=False, explanation="Abuse pattern detected"
                  )

          # Check model response for labels
          if "frustration" in response_lower or "complaint" in response_lower:
              return ContentAnalysis(query, "frustration", "parsed", 0.7, False, "From response")
          if "abusive" in response_lower or "harassment" in response_lower:
              return ContentAnalysis(query, "abusive", "parsed", 0.7, False, "From response")
          if "severe" in response_lower:
              return ContentAnalysis(query, "severe", "parsed", 0.8, False, "From response")

          # Default to clean
          return ContentAnalysis(query, "clean", "none", 0.6, True, "Default classification")

      @torch.no_grad()
      def filter_batch(self, queries: List[str]) -> List[ContentAnalysis]:
          """Process a batch of queries."""

          prompts = [self.prompt_template.format(query=q) for q in queries]

          inputs = self.tokenizer(
              prompts,
              return_tensors="pt",
              padding=True,
              truncation=True,
              max_length=768
          ).to(self.model.device)

          outputs = self.model.generate(
              **inputs,
              max_new_tokens=80,
              temperature=0.1,
              do_sample=False,
              pad_token_id=self.tokenizer.pad_token_id,
              eos_token_id=self.tokenizer.eos_token_id,
          )

          input_len = inputs["input_ids"].shape[1]
          responses = self.tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

          results = []
          for query, response in zip(queries, responses):
              analysis = self._parse_response(response, query)
              results.append(analysis)

          return results

      def filter_all(
          self,
          queries: List[str],
          show_progress: bool = True
      ) -> pd.DataFrame:
          """Filter all queries."""

          all_results = []
          num_batches = math.ceil(len(queries) / self.batch_size)

          iterator = range(num_batches)
          if show_progress:
              iterator = tqdm(iterator, desc="Filtering content")

          for i in iterator:
              start = i * self.batch_size
              end = min(start + self.batch_size, len(queries))
              batch = queries[start:end]

              if not batch:
                  continue

              try:
                  results = self.filter_batch(batch)
                  all_results.extend(results)
              except Exception as e:
                  print(f"Batch {i} error: {e}")
                  for q in batch:
                      all_results.append(ContentAnalysis(
                          query=q, label="error", category="error",
                          confidence=0.0, keep=False, explanation=str(e)[:50]
                      ))

              if i % 5 == 0:
                  torch.cuda.empty_cache()

          return pd.DataFrame([r.to_dict() for r in all_results])

      def process_dataframe(
          self,
          df: pd.DataFrame,
          query_column: str = "query",
          output_path: str = None
      ) -> pd.DataFrame:
          """Process DataFrame and return filtered results."""

          queries = df[query_column].dropna().astype(str).tolist()

          print(f"\nProcessing {len(queries):,} queries...")
          start_time = time.time()

          result_df = self.filter_all(queries)

          elapsed = time.time() - start_time

          # Statistics
          print(f"\n{'='*60}")
          print("CONTENT FILTERING RESULTS")
          print(f"{'='*60}")
          print(f"Total queries:    {len(result_df):,}")
          print(f"Processing time:  {elapsed:.1f}s ({len(queries)/elapsed:.1f} qps)")

          # Label breakdown
          print(f"\nLabel Distribution:")
          for label in ["clean", "frustration", "mild", "abusive", "severe"]:
              count = (result_df["label"] == label).sum()
              pct = 100 * count / len(result_df) if len(result_df) > 0 else 0
              status = "✅ KEEP" if label == "clean" else "❌ REMOVE"
              print(f"  {label.upper():12s}: {count:6,} ({pct:5.2f}%) {status}")

          # Keep vs Remove
          keep_count = result_df["keep"].sum()
          remove_count = len(result_df) - keep_count

          print(f"\nFinal Summary:")
          print(f"  ✅ Keep (clean):     {keep_count:,} ({100*keep_count/len(result_df):.2f}%)")
          print(f"  ❌ Remove (bad):     {remove_count:,} ({100*remove_count/len(result_df):.2f}%)")

          # Category breakdown for removed
          removed_df = result_df[~result_df["keep"]]
          if len(removed_df) > 0:
              print(f"\nRemoved Categories:")
              cat_counts = removed_df["category"].value_counts()
              for cat, count in cat_counts.head(10).items():
                  print(f"    {cat}: {count}")

          if output_path:
              result_df.to_csv(output_path, index=False)
              print(f"\nSaved full results to: {output_path}")

              # Also save clean queries separately
              clean_path = output_path.replace(".csv", "_clean.csv")
              clean_df = result_df[result_df["keep"]][["query"]]
              clean_df.to_csv(clean_path, index=False)
              print(f"Saved clean queries to: {clean_path}")

          return result_df


  # =============================================================================
  # 4. SIMPLE FUNCTION FOR QUICK USE
  # =============================================================================

  def filter_queries_for_training(
      input_csv: str,
      output_csv: str,
      query_column: str = "query",
      model_name: str = "google/gemma-3-12b-it",
      batch_size: int = 8
  ) -> pd.DataFrame:
      """
      Filter queries: Remove abuse AND frustration.
      Keep only clean queries for autocomplete training.
      
      Args:
          input_csv: Input CSV path
          output_csv: Output CSV path (full results)
          query_column: Column name with queries
          model_name: Model to use
          batch_size: Batch size (reduce if OOM)
      
      Returns:
          DataFrame with all queries labeled
          Also saves {output_csv}_clean.csv with only clean queries
      """

      df = pd.read_csv(input_csv)

      filter_model = ContentFilter(
          model_name=model_name,
          prompt_type="efficient",
          batch_size=batch_size
      )

      result_df = filter_model.process_dataframe(df, query_column, output_csv)

      return result_df


  # =============================================================================
  # 5. MAIN
  # =============================================================================

  if __name__ == "__main__":
      # Test queries covering all categories
      test_queries = [
          # Clean - KEEP ✅
          "what is my account balance",
          "transfer money to savings",
          "check my credit card limit",
          "pay my bill",
          "activate my new card",
          "how to set up direct deposit",
          "view recent transactions",
          "change my password",
          "find nearest ATM",
          "apply for a loan",

          # Frustration - REMOVE ❌
          "why is this so slow",
          "this is frustrating",
          "ugh nothing works",
          "your app sucks",
          "I hate this bank",
          "worst service ever",
          "so annoying",
          "why can't I do anything",
          "this is ridiculous",
          "waste of time",
          "disappointed with this",
          "useless app",

          # Mild inappropriate - REMOVE ❌
          "damn this is hard",
          "what the hell is wrong",
          "crap it's not working",

          # Abusive - REMOVE ❌
          "you people are idiots",
          "incompetent morons",
          "stupid bank employees",
          "f*** this app",

          # Severe - REMOVE ❌
          "I'll burn this place down",
          "you [slur] don't know anything",
          "how to hack accounts",
      ]

      # Create test CSV
      test_df = pd.DataFrame({"query": test_queries})
      test_df.to_csv("/tmp/test_queries.csv", index=False)

      # Run filter
      result_df = filter_queries_for_training(
          input_csv="/tmp/test_queries.csv",
          output_csv="/tmp/filtered_results.csv",
          batch_size=4
      )

      # Show detailed results
      print("\n" + "="*80)
      print("DETAILED RESULTS")
      print("="*80)

      for label in ["clean", "frustration", "mild", "abusive", "severe"]:
          subset = result_df[result_df["label"] == label]
          if len(subset) > 0:
              status = "✅ KEEP" if label == "clean" else "❌ REMOVE"
              print(f"\n--- {label.upper()} ({len(subset)}) {status} ---")
              for _, row in subset.head(5).iterrows():
                  print(f"  [{row['confidence']:.2f}] {row['query'][:50]}")
                  print(f"         → {row['category']}: {row['explanation'][:40]}")

